# PM4Py Process Mining on UNRDF OTEL Logs

This notebook extracts event logs from Loki and applies process mining techniques:
1. **Log Extraction** — Query Loki for OTEL span events
2. **Event Log Construction** — Convert traces to process mining format
3. **Process Discovery** — Discover BPMN and Petri net models
4. **Conformance Checking** — Compare actual vs expected behavior
5. **Bottleneck Analysis** — Identify performance bottlenecks

In [ ]:
# Papermill parameters (overridable via CLI)
# Usage: papermill pm4py-process-mining.ipynb output.ipynb -p hours 6 -p limit 500
hours = 24  # Time range in hours to query
limit = 1000  # Max traces/logs to fetch
service_filter = ""  # Filter by service name (empty = all)

import pm4py
import requests
import pandas as pd
import json
import re
from datetime import datetime, timedelta
import os

LOKI_URL = os.environ.get("LOKI_URL", "http://loki:3100")
TEMPO_URL = os.environ.get("TEMPO_URL", "http://tempo:3200")
PROMETHEUS_URL = os.environ.get("PROMETHEUS_URL", "http://prometheus:9090")

## 1. Extract Traces from Tempo
Query Tempo API for traces and convert to event log format.

In [ ]:
def fetch_traces_from_tempo(hours=1, limit=1000):
    """Fetch traces from Tempo and return as DataFrame."""
    end = int(datetime.utcnow().timestamp() * 1e9)
    start = int((datetime.utcnow() - timedelta(hours=hours)).timestamp() * 1e9)
    
    # Search for all traces in the time range
    url = f"{TEMPO_URL}/api/search?start={start}&end={end}&limit={limit}"
    resp = requests.get(url)
    resp.raise_for_status()
    
    traces = resp.json().get('traces', [])
    events = []
    
    for trace in traces:
        trace_id = trace['traceID']
        
        # Fetch full trace
        trace_url = f"{TEMPO_URL}/api/traces/{trace_id}"
        trace_resp = requests.get(trace_url)
        if trace_resp.status_code != 200:
            continue
        
        trace_data = trace_resp.json()
        for batch in trace_data.get('batches', []):
            for scope_span in batch.get('scopeSpans', []):
                for span in scope_span.get('spans', []):
                    attrs = {}
                    for a in span.get('attributes', []):
                        v = a.get('value', {})
                        attrs[a['key']] = v.get('stringValue') or v.get('intValue') or str(v)
                    
                    events.append({
                        'case:concept:name': trace_id,
                        'concept:name': span['name'],
                        'time:timestamp': datetime.fromtimestamp(
                            int(span['startTimeUnixNano']) / 1e9
                        ).isoformat(),
                        'org:resource': attrs.get('service.name', 'unknown'),
                        'duration_ms': (int(span.get('endTimeUnixNano', 0)) - int(span.get('startTimeUnixNano', 0))) / 1e6,
                        'span_id': span['spanId'],
                        'mcp.tool.name': attrs.get('mcp.tool.name', ''),
                        'mcp.tool.success': attrs.get('mcp.tool.success', ''),
                    })
    
    return pd.DataFrame(events)

df = fetch_traces_from_tempo(hours=24)
print(f"Extracted {len(df)} events from {df['case:concept:name'].nunique()} traces")
df.head(10)

## 2. Extract Logs from Loki
Query Loki for container logs and convert to event log format.

In [ ]:
def fetch_logs_from_loki(hours=1, limit=1000):
    """Fetch logs from Loki and return as DataFrame."""
    end = int(datetime.utcnow().timestamp())
    start = int((datetime.utcnow() - timedelta(hours=hours)).timestamp())
    
    query = '{container=~"unrdf.*"}'
    url = f"{LOKI_URL}/loki/api/v1/query_range"
    params = {
        'query': query,
        'start': str(start) + '000000000',
        'end': str(end) + '000000000',
        'limit': limit,
        'direction': 'forward',
    }
    
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    
    data = resp.json()
    events = []
    
    for result in data.get('data', {}).get('result', []):
        container = result.get('stream', {}).get('container', 'unknown')
        for value in result.get('values', []):
            ts_ns, line = value
            ts = datetime.fromtimestamp(int(ts_ns) / 1e9).isoformat()
            
            # Extract trace ID if present
            trace_id = ''
            if '"traceID"' in line:
                import re
                m = re.search(r'"traceID":"([a-f0-9]+)"', line)
                if m:
                    trace_id = m.group(1)
            
            events.append({
                'case:concept:name': trace_id or container,
                'concept:name': 'log_entry',
                'time:timestamp': ts,
                'org:resource': container,
                'log_level': 'info' if 'info' in line.lower() else ('error' if 'error' in line.lower() else 'debug'),
                'message': line[:200],
            })
    
    return pd.DataFrame(events)

log_df = fetch_logs_from_loki(hours=24)
print(f"Extracted {len(log_df)} log entries")
log_df.head(10)

## 3. Process Discovery — Inductive Miner
Discover a process model from the event log.

In [ ]:
# Combine trace events and log events
if len(df) > 0:
    # Convert timestamp
    df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])
    
    # Discover process tree using Inductive Miner
    net, im, fm = pm4py.discover_petri_net_inductive(df)
    print(f"Places: {len(net.places)}, Transitions: {len(net.transitions)}, Arcs: {len(net.arcs)}")
    
    # Visualize
    pm4py.view_petri_net(net, im, fm)
else:
    print("No trace events found. Generate traces first using the HotROD demo app.")
    print("Open http://localhost:8080 and click around to generate traces.")

## 4. BPMN Process Model Discovery

In [ ]:
if len(df) > 0:
    bpmn_model = pm4py.discover_bpmn_inductive(df)
    pm4py.view_bpmn(bpmn_model)
else:
    print("No events to discover model from.")

## 5. Conformance Checking
Check how well traces conform to the discovered model.

In [ ]:
if len(df) > 0:
    # Token-based replay fitness
    fitness = pm4py.fitness_token_based_replay(df, net, im, fm)
    print(f"Fitness: {fitness}")
    
    # Precision
    precision = pm4py.precision_token_based_replay(df, net, im, fm)
    print(f"Precision: {precision}")
else:
    print("No events for conformance checking.")

## 6. Bottleneck Analysis
Identify the slowest activities in the process.

In [ ]:
if len(df) > 0 and 'duration_ms' in df.columns:
    # Activity duration analysis
    activity_stats = df.groupby('concept:name')['duration_ms'].agg(['mean', 'median', 'max', 'count'])
    activity_stats = activity_stats.sort_values('mean', ascending=False)
    print("Activity Duration Analysis (ms):")
    print(activity_stats.to_string())
    
    # Case duration
    case_duration = df.groupby('case:concept:name').agg(
        start=('time:timestamp', 'min'),
        end=('time:timestamp', 'max'),
        span_count=('concept:name', 'count')
    )
    case_duration['duration'] = (case_duration['end'] - case_duration['start']).dt.total_seconds()
    print(f"\nCase Duration: mean={case_duration['duration'].mean():.2f}s, max={case_duration['duration'].max():.2f}s")
else:
    print("No events for bottleneck analysis.")

## 7. Process Map Visualization
Visualize the process flow as a frequency map.

In [ ]:
if len(df) > 0:
    pm4py.view_process_tree(pm4py.discover_process_tree_inductive(df))
else:
    print("No events for visualization.")

## 8. Temporal Profile Conformance
Check SLA compliance at the process level — compare observed inter-activity times against expected temporal profiles.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import numpy as np

        # Discover temporal profile (expected time between activity pairs)
        temporal_profile = pm4py.discover_temporal_profile(df)

        # Check conformance against temporal profile
        conformance = pm4py.conformance_temporal_profile(df, temporal_profile)

        # Aggregate deviations (zeta_score > 2.0 = significant)
        total_checks = 0
        total_deviations = 0
        worst = []

        for trace_conformance in conformance:
            for check in trace_conformance:
                total_checks += 1
                if isinstance(check, (list, tuple)) and len(check) >= 4:
                    act1, act2 = str(check[0]), str(check[1])
                    observed = float(check[2])
                    zeta = float(check[3])
                    if zeta > 2.0:
                        total_deviations += 1
                        if len(worst) < 5:
                            worst.append(f"{act1} -> {act2}: {observed:.3f}s (zeta={zeta:.2f})")

        # Most variable pairs from the temporal profile
        variable_pairs = sorted(temporal_profile.items(), key=lambda x: x[1][1], reverse=True)[:5]

        print(f"Temporal Profile Conformance:")
        print(f"  Activity pairs: {len(temporal_profile)}")
        print(f"  Total checks: {total_checks}")
        print(f"  Significant deviations (zeta > 2.0): {total_deviations} ({100*total_deviations/max(total_checks,1):.1f}%)")
        print(f"\n  Most variable pairs (by std):")
        for (a1, a2), (mean_t, std_t) in variable_pairs:
            print(f"    {a1} -> {a2}: mean={mean_t*1000:.1f}ms, std={std_t*1000:.1f}ms")
        if worst:
            print(f"\n  Worst SLA violations:")
            for w in worst:
                print(f"    {w}")
    except Exception as e:
        print(f"Temporal profile conformance failed: {e}")
else:
    print("No events for temporal profile conformance.")

## 9. Rework Detection
Identify activities that occur multiple times within the same trace, indicating rework loops or retry patterns.

In [ ]:
if len(df) > 0:
    try:
        import pm4py

        # Get rework statistics per activity
        rework_cases = pm4py.get_rework_cases_per_activity(df)
        sorted_rework = sorted(rework_cases.items(), key=lambda x: x[1], reverse=True)

        rework_details = []
        for activity, count in sorted_rework[:5]:
            if count > 0:
                rework_df = pm4py.filter_activities_rework(df, activity, min_occurrences=2)
                rework_trace_count = rework_df['case:concept:name'].nunique() if len(rework_df) > 0 else 0
                rework_rate = 100 * rework_trace_count / max(df['case:concept:name'].nunique(), 1)
                rework_details.append((activity, count, rework_trace_count, rework_rate))

        total_rework = sum(c for _, c in sorted_rework)
        activities_with_rework = sum(1 for _, c in sorted_rework if c > 0)

        print(f"Rework Detection:")
        print(f"  Activities with rework: {activities_with_rework}")
        print(f"  Total rework cases: {total_rework}")
        print(f"\n  Top reworked activities:")
        for activity, cases, traces, rate in rework_details:
            print(f"    {activity:45s} {cases:4d} cases, {traces:3d} traces ({rate:.1f}%)")
    except Exception as e:
        print(f"Rework detection failed: {e}")
else:
    print("No events for rework detection.")

## 10. Batch Detection
Detect batch processing patterns — multiple cases processed together — that explain tail latency spikes.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import numpy as np

        # Discover batches (large merge_distance for distributed traces)
        batches = pm4py.discover_batches(df, merge_distance=30000, min_batch_size=2)

        if batches:
            batch_sizes = [b[1] for b in batches]
            print(f"Batch Detection:")
            print(f"  Batches found: {len(batches)}")
            print(f"  Total batched cases: {sum(batch_sizes)}")
            print(f"  Avg batch size: {np.mean(batch_sizes):.1f}")
            print(f"  Max batch size: {max(batch_sizes)}")
        else:
            print("No formal batches detected.")
            print("Note: Distributed traces may not exhibit classical batching patterns.")
    except Exception as e:
        print(f"Batch detection failed: {e}")
else:
    print("No events for batch detection.")

## 11. Social Network Analysis
Discover service interaction patterns — handover-of-work and working-together networks.

In [ ]:
if len(df) > 0:
    n_resources = df['org:resource'].nunique()
    if n_resources < 2:
        print(f"SNA skipped: need >= 2 distinct resources, found {n_resources}")
        print("Ensure traces include service.name attribute for SNA.")
    else:
        try:
            import pm4py

            # Handover-of-work network
            how = pm4py.discover_handover_of_work_network(df)
            how_edges = list(how.connections) if hasattr(how, 'connections') else []

            # Working-together network
            wt = pm4py.discover_working_together_network(df)
            wt_edges = list(wt.connections) if hasattr(wt, 'connections') else []

            # Organizational roles
            roles = pm4py.discover_organizational_roles(df)

            print(f"Social Network Analysis:")
            print(f"  Handover-of-work edges: {len(how_edges)}")
            print(f"  Working-together edges: {len(wt_edges)}")
            print(f"  Organizational roles: {len(roles) if roles else 0}")

            if how_edges:
                print(f"\n  Top handovers:")
                for src, tgt, w in sorted(how_edges, key=lambda x: x[2] if len(x)>2 else 0, reverse=True)[:5]:
                    print(f"    {src} -> {tgt}: weight={w}")

            if roles:
                print(f"\n  Roles:")
                for r in roles[:5]:
                    name = r.name if hasattr(r, 'name') else str(r)
                    acts = len(r.activities) if hasattr(r, 'activities') else 0
                    print(f"    {name}: {acts} activities")
        except Exception as e:
            print(f"SNA failed: {e}")
else:
    print("No events for social network analysis.")

## 12. Decision Mining
Discover data-driven routing decisions at gateway points using alignment features + decision tree classification.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import pandas as pd
        from sklearn.tree import DecisionTreeClassifier, export_text

        # Get alignments with diagnostic info
        aligned = pm4py.conformance_diagnostics_alignments(df, net, im, fm)

        trace_features = []
        trace_labels = []

        for alignment in aligned:
            if isinstance(alignment, dict):
                features = {
                    'alignment_cost': alignment.get('cost', 0),
                    'fitness': alignment.get('fitness', 0),
                }
                alignment_data = alignment.get('alignment', [])
                features['log_moves'] = sum(1 for m in alignment_data if isinstance(m, tuple) and m[0] is None)
                features['model_moves'] = sum(1 for m in alignment_data if isinstance(m, tuple) and m[1] is None)
                trace_labels.append(alignment.get('trace_is_fit', True))
                trace_features.append(features)

        if len(trace_features) >= 5:
            feat_df = pd.DataFrame(trace_features)
            labels = pd.Series(trace_labels)

            if labels.nunique() >= 2:
                clf = DecisionTreeClassifier(max_depth=3, random_state=42)
                clf.fit(feat_df, labels)
                tree_rules = export_text(clf, feature_names=list(feat_df.columns))

                print(f"Decision Mining:")
                print(f"  Tree depth: {clf.get_depth()}")
                print(f"  Feature importance: {dict(zip(feat_df.columns, clf.feature_importances_.round(3)))}")
                print(f"  Fit traces: {int(labels.sum())}, Unfit: {int((~labels).sum())}")
                print(f"\n  Decision tree rules:\n{tree_rules}")
            else:
                print(f"Decision Mining: insufficient class diversity (all traces are {labels.iloc[0]})")
        else:
            print("Decision Mining: insufficient traces for analysis")
    except Exception as e:
        print(f"Decision mining failed: {e}")
else:
    print("No events for decision mining.")

## 13. Performance Spectrum Analysis
Analyze activity latency distributions to detect bimodal patterns and outliers.

In [ ]:
if len(df) > 0 and 'duration_ms' in df.columns:
    try:
        import numpy as np

        activity_counts = df['concept:name'].value_counts().head(10)
        spectrum_results = []

        for activity, count in activity_counts.items():
            durations = df[df['concept:name'] == activity]['duration_ms']

            if len(durations) >= 5:
                q25, q75 = durations.quantile(0.25), durations.quantile(0.75)
                iqr = q75 - q25
                outlier_count = int(((durations < q25 - 1.5*iqr) | (durations > q75 + 1.5*iqr)).sum())
                cv = durations.std() / durations.mean() if durations.mean() > 0 else 0

                spectrum_results.append({
                    'activity': activity,
                    'count': int(count),
                    'mean': float(durations.mean()),
                    'median': float(durations.median()),
                    'p95': float(durations.quantile(0.95)),
                    'p99': float(durations.quantile(0.99)),
                    'std': float(durations.std()),
                    'cv': float(cv),
                    'outliers': outlier_count,
                    'bimodal': cv > 0.8 or outlier_count > len(durations) * 0.1,
                })

        spectrum_results.sort(key=lambda x: x['cv'], reverse=True)

        print(f"Performance Spectrum Analysis:")
        print(f"  Activities analyzed: {len(spectrum_results)}")
        print(f"  Bimodal activities: {sum(1 for s in spectrum_results if s['bimodal'])}")
        print(f"\n  {'Activity':45s} {'Count':>5s} {'Mean':>8s} {'P95':>8s} {'P99':>8s} {'CV':>6s} {'Out':>4s}")
        print(f"  {'-'*90}")
        for s in spectrum_results[:10]:
            flag = ' *' if s['bimodal'] else ''
            print(f"  {s['activity']:45s} {s['count']:5d} {s['mean']:8.1f} {s['p95']:8.1f} {s['p99']:8.1f} {s['cv']:6.3f} {s['outliers']:4d}{flag}")
        print(f"\n  * = bimodal (CV > 0.8 or outlier rate > 10%)")
    except Exception as e:
        print(f"Performance spectrum analysis failed: {e}")
else:
    print("No events for performance spectrum analysis.")

## 14. Declare (Declarative Process Mining)

Discover Linear Temporal Logic (LTL) constraints that describe the *de facto* rules of the process. Unlike imperative models (Petri nets), declarative mining expresses what activities **must, must not, or may** occur relative to each other — ideal for flexible, low-structure processes like distributed trace execution.

- **Constraint discovery** — `pm4py.discover_declare()` scans the log for frequently satisfied LTL templates (e.g., *existence*, *absence*, *response*, *precedence*, *succession*).
- **Conformance checking** — `pm4py.conformance_declare()` evaluates each trace against the discovered constraint set, reporting per-trace fitness and specific violations.

The `min_support_ratio` parameter controls the minimum fraction of traces that must satisfy a template for it to be included.

In [ ]:
if len(df) > 0:
    try:
        import pm4py

        # Discover declarative (LTL) constraints from the log
        declare_model = pm4py.discover_declare(df, min_support_ratio=0.5)

        # Summarize discovered constraints
        total_constraints = len(declare_model)
        template_counts = {}
        for tpl, instances in declare_model.items():
            template_name = tpl if isinstance(tpl, str) else getattr(tpl, 'name', str(tpl))
            template_counts[template_name] = template_counts.get(template_name, 0) + 1

        print(f"Declarative Constraint Discovery:")
        print(f"  Total constraints discovered: {total_constraints}")
        print(f"  Constraint templates used: {len(template_counts)}")
        print(f"\n  Template distribution:")
        for tpl, count in sorted(template_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"    {tpl}: {count}")

        # Conformance checking against discovered constraints
        conformance = pm4py.conformance_declare(df, declare_model)

        if conformance:
            fitnesses = [c.get('dev_fitness', 0) for c in conformance if isinstance(c, dict)]
            deviating = [c for c in conformance if isinstance(c, dict) and c.get('dev_fitness', 1.0) < 1.0]

            print(f"\n  Conformance Results:")
            print(f"    Mean fitness:   {sum(fitnesses)/len(fitnesses):.4f}")
            print(f"    Deviating traces: {len(deviating)}/{len(conformance)}")

            if deviating:
                print(f"\n  Top 5 violations:")
                for i, d in enumerate(sorted(deviating, key=lambda x: x.get('dev_fitness', 1.0))[:5]):
                    violations = d.get('deviations', [])
                    print(f"    {i+1}. fitness={d.get('dev_fitness', 'N/A')}, violations={len(violations)}")
                    for v in violations[:2]:
                        print(f"       - {v}")
        else:
            print("\n  No conformance results returned")
    except Exception as e:
        print(f"Declarative mining failed: {e}")
else:
    print("No events for declarative process mining.")

## 15. Event Data Quality

Assess the fitness of the event log for process mining along four ISO-inspired dimensions:

| Dimension | What it measures |
|-----------|-----------------|
| **Completeness** | Are there missing case IDs, activities, or timestamps? |
| **Validity** | Do timestamps parse correctly? Are required columns present? |
| **Timeliness** | Are events in chronological order per trace? |
| **Coverage** | Does the log cover the expected time range with sufficient trace density? |

Low-quality event data produces misleading process models. This section produces a diagnostic summary so you can decide whether to filter, repair, or re-extract before downstream analysis.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import pandas as pd
        import numpy as np

        total_events = len(df)
        total_cases = df['case:concept:name'].nunique()
        issues = []

        # 1. Completeness — missing values in core columns
        core_cols = ['case:concept:name', 'concept:name', 'time:timestamp']
        missing = {}
        for col in core_cols:
            if col in df.columns:
                n_missing = int(df[col].isna().sum())
                missing[col] = n_missing
                if n_missing > 0:
                    issues.append(f"{col}: {n_missing} missing values ({100*n_missing/total_events:.1f}%)")
            else:
                missing[col] = total_events
                issues.append(f"{col}: column not present")

        completeness_score = 1.0 - sum(missing.values()) / (total_events * len(core_cols))

        # 2. Validity — timestamp parsing and required columns
        validity_issues = 0
        if 'time:timestamp' in df.columns:
            invalid_ts = 0
            for val in df['time:timestamp']:
                try:
                    pd.to_datetime(val)
                except Exception:
                    invalid_ts += 1
            if invalid_ts > 0:
                validity_issues += invalid_ts
                issues.append(f"time:timestamp: {invalid_ts} unparseable values")

        validity_score = 1.0 - validity_issues / max(total_events, 1)

        # 3. Timeliness — events in chronological order per case
        df_sorted = df.sort_values(['case:concept:name', 'time:timestamp'])
        out_of_order = 0
        for case_id, group in df.groupby('case:concept:name'):
            ts = pd.to_datetime(group['time:timestamp'].values)
            if not pd.Series(ts).is_monotonic_increasing:
                out_of_order += 1

        timeliness_score = 1.0 - out_of_order / max(total_cases, 1)
        if out_of_order > 0:
            issues.append(f"chronological order: {out_of_order}/{total_cases} traces have out-of-order events")

        # 4. Coverage — time range and trace density
        if 'time:timestamp' in df.columns:
            ts = pd.to_datetime(df['time:timestamp'])
            time_range_hours = (ts.max() - ts.min()).total_seconds() / 3600
            density = total_cases / max(time_range_hours, 0.001)
            coverage_score = min(density / 10.0, 1.0)  # 10+ cases/hour = full coverage
            if density < 1.0:
                issues.append(f"coverage: {density:.1f} cases/hour (< 1.0 = sparse)")
        else:
            coverage_score = 0.0
            time_range_hours = 0
            density = 0

        overall = (completeness_score + validity_score + timeliness_score + coverage_score) / 4

        print("Event Data Quality Assessment:")
        print(f"  Events: {total_events}, Cases: {total_cases}")
        print(f"  Time range: {time_range_hours:.1f} hours")
        print(f"  Trace density: {density:.1f} cases/hour")
        print(f"\n  Dimension Scores:")
        print(f"    Completeness: {completeness_score:.4f}")
        print(f"    Validity:     {validity_score:.4f}")
        print(f"    Timeliness:   {timeliness_score:.4f}")
        print(f"    Coverage:     {coverage_score:.4f}")
        print(f"    OVERALL:      {overall:.4f}")

        if issues:
            print(f"\n  Issues ({len(issues)}):")
            for issue in issues:
                print(f"    - {issue}")
        else:
            print("\n  No quality issues detected.")
    except Exception as e:
        print(f"Data quality assessment failed: {e}")
else:
    print("No events for data quality assessment.")

## 16. Process Simulation

Generate synthetic traces from the discovered process model using `pm4py.play_out()`. This enables **what-if analysis** — you can simulate expected behavior under normal conditions and compare it against observed production traces.

- **Play-out** — Stochastically walks the Petri net (from Section 3) to produce realistic trace sequences.
- **Use cases** — Baseline comparison, capacity planning, anomaly calibration, and stress-testing conformance checkers.

The simulated log preserves the structural properties of the discovered model (parallelism, choice, loops) while generating activity timestamps that reflect observed distributions.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import pandas as pd

        # Generate synthetic traces from the discovered Petri net
        simulated_log = pm4py.play_out(net, im, fm, num_traces=100)

        # Convert simulated log to DataFrame for analysis
        sim_df = pm4py.convert_to_dataframe(simulated_log)

        sim_cases = sim_df['case:concept:name'].nunique()
        sim_events = len(sim_df)
        sim_activities = sim_df['concept:name'].nunique()

        # Variant distribution in simulated log
        sim_variants = pm4py.get_variants(sim_df)

        # Compare with observed variants
        obs_variants = pm4py.get_variants(df)
        common_variants = set(sim_variants.keys()) & set(obs_variants.keys())
        sim_only = set(sim_variants.keys()) - set(obs_variants.keys())

        print(f"Process Simulation (play-out):")
        print(f"  Simulated cases:  {sim_cases}")
        print(f"  Simulated events: {sim_events}")
        print(f"  Unique activities: {sim_activities}")
        print(f"  Simulated variants: {len(sim_variants)}")
        print(f"  Observed variants:  {len(obs_variants)}")
        print(f"  Common variants:    {len(common_variants)}")
        print(f"  Sim-only variants:  {len(sim_only)}")

        if sim_only:
            print(f"\n  Simulated-only variants (top 5):")
            for v in sorted(sim_only, key=lambda x: sim_variants[x], reverse=True)[:5]:
                print(f"    - {' \u2192 '.join(v)}")

        # Conformance of simulated log against the model (should be high)
        sim_fitness = pm4py.fitness_token_based_replay(sim_df, net, im, fm)
        print(f"\n  Simulated log fitness: {sim_fitness}")
    except Exception as e:
        print(f"Process simulation failed: {e}")
else:
    print("No events for process simulation.")

## 17. Process Cube

Perform OLAP-style multi-dimensional analysis on the event log. The **Process Cube** slices the log along two dimensions (e.g., time period and resource) and aggregates a target metric (e.g., case duration or activity count).

- **Feature extraction** — Enriches the DataFrame with outcome labels and aggregate features per case.
- **Cube construction** — Groups the log by `(x_col, y_col)` and computes the specified aggregation.

This is useful for answering questions like: *"Do cases handled by service X take longer on Mondays?"* or *"Has average trace latency changed week over week?"*

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import pandas as pd

        # Enrich the DataFrame with outcome labels and case-level features
        enriched = pm4py.extract_outcome_enriched_dataframe(df)

        # Extract features (case-level aggregates)
        features = pm4py.extract_features_dataframe(enriched)

        # Determine available columns for cube dimensions
        x_col = None
        y_col = None

        if 'time:timestamp' in features.columns:
            features['_hour'] = pd.to_datetime(features['time:timestamp']).dt.hour
            x_col = '_hour'

        if 'org:resource' in features.columns:
            y_col = 'org:resource'

        if x_col and y_col:
            agg_col = 'case:concept:name'  # count traces as default aggregation
            cube_df, cube_config = pm4py.get_process_cube(features, x_col, y_col, agg_col)

            print(f"Process Cube Analysis:")
            print(f"  X dimension: {x_col}")
            print(f"  Y dimension: {y_col}")
            print(f"  Aggregation: count of {agg_col}")
            print(f"  Cube cells:  {len(cube_df)}")
            print(f"\n  {cube_df.to_string()}")
        else:
            # Fallback: simple group-by analysis
            print("Process Cube (fallback — group-by analysis):")
            if 'org:resource' in features.columns:
                by_resource = features.groupby('org:resource').size().sort_values(ascending=False)
                print(f"\n  Cases per resource:")
                for res, count in by_resource.head(10).items():
                    print(f"    {res}: {count}")
            else:
                print("  Insufficient columns for multi-dimensional analysis")
                print(f"  Available columns: {list(features.columns)}")
    except Exception as e:
        print(f"Process cube analysis failed: {e}")
else:
    print("No events for process cube analysis.")

## 18. Streaming Discovery

Process discovery on **event streams** for incremental, real-time model updates. Instead of batch mining the entire log, the streaming discovery algorithm maintains a running Directly-Follows Graph (DFG) that is updated as each event arrives.

- **StreamingDfgDiscovery** — Processes events one at a time via `.receive_event()`, maintaining frequency counts.
- **Incremental DFG** — Call `.check()` at any point to retrieve the current DFG without reprocessing.

This pattern is designed for live telemetry pipelines where the full log is not available upfront — e.g., a sidecar that updates the process model as OTEL spans arrive from Loki.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        from pm4py.streaming.algo.discovery.dfg.variants.frequency import StreamingDfgDiscovery

        # Convert DataFrame to event stream for streaming ingestion
        stream = pm4py.convert_to_event_stream(df)

        # Instantiate streaming DFG discovery
        stream_disc = StreamingDfgDiscovery()

        # Feed events one at a time
        event_count = 0
        for event in stream:
            stream_disc.receive(event)
            event_count += 1

        # Retrieve the discovered DFG
        streaming_dfg = stream_disc.check()

        # Analyze the streaming DFG
        dfg_graph = streaming_dfg.dfg if hasattr(streaming_dfg, 'dfg') else streaming_dfg
        start_activities = streaming_dfg.start_activities if hasattr(streaming_dfg, 'start_activities') else {}
        end_activities = streaming_dfg.end_activities if hasattr(streaming_dfg, 'end_activities') else {}

        # Sort edges by frequency
        sorted_edges = sorted(dfg_graph.items(), key=lambda x: x[1], reverse=True) if isinstance(dfg_graph, dict) else []

        print(f"Streaming DFG Discovery:")
        print(f"  Events processed: {event_count}")
        print(f"  DFG edges: {len(sorted_edges)}")
        print(f"  Start activities: {len(start_activities)}")
        print(f"  End activities: {len(end_activities)}")
        print(f"\n  Top 15 edges by frequency:")
        for (src, tgt), freq in sorted_edges[:15]:
            print(f"    {str(src):40s} -> {str(tgt):40s} : {freq}")

        # Compare with batch DFG
        batch_dfg, batch_sa, batch_ea = pm4py.discover_dfg(df)
        batch_edges = set(batch_dfg.keys())
        stream_edges = set(dfg_graph.keys()) if isinstance(dfg_graph, dict) else set()
        print(f"\n  Comparison with batch DFG:")
        print(f"    Batch edges: {len(batch_edges)}")
        print(f"    Stream edges: {len(stream_edges)}")
        print(f"    Matching: {len(batch_edges & stream_edges)}")
        print(f"    Stream-only: {len(stream_edges - batch_edges)}")
        print(f"    Batch-only: {len(batch_edges - stream_edges)}")
    except Exception as e:
        print(f"Streaming discovery failed: {e}")
else:
    print("No events for streaming discovery.")

## 19. Act — Automated Remediation

Close the Observe-Detect-Check-Act loop by translating conformance diagnostics into **actionable remediation recommendations**. This section:

1. **Runs Declare conformance** to identify constraint violations per trace.
2. **Identifies low-fitness traces** — traces with fitness below a configurable threshold.
3. **Outputs remediation recommendations** — classifies violations by severity and suggests corrective actions.

The recommendations are structured as a JSON-serializable report that can be consumed by downstream automation (e.g., alerting, ticket creation, or runbook triggering).

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import json
        from datetime import datetime

        FITNESS_THRESHOLD = 0.8

        # Step 1: Discover Declare constraints
        declare_model = pm4py.discover_declare(df, min_support_ratio=0.5)

        # Step 2: Run conformance checking
        conformance = pm4py.conformance_declare(df, declare_model)

        if not conformance:
            print("No conformance results — all traces may satisfy all constraints.")
        else:
            # Step 3: Identify low-fitness traces
            low_fitness = []
            recommendations = []

            for i, result in enumerate(conformance):
                if not isinstance(result, dict):
                    continue

                fitness = result.get('dev_fitness', 1.0)
                deviations = result.get('deviations', [])

                if fitness < FITNESS_THRESHOLD:
                    # Get trace identifier
                    case_ids = df['case:concept:name'].unique()
                    case_id = case_ids[i] if i < len(case_ids) else f"trace_{i}"

                    severity = "critical" if fitness < 0.5 else ("warning" if fitness < 0.8 else "info")

                    # Classify deviation types for remediation
                    deviation_types = set()
                    for dev in deviations:
                        dev_str = str(dev).lower()
                        if 'response' in dev_str or 'succession' in dev_str:
                            deviation_types.add('ordering')
                        elif 'absence' in dev_str or 'negation' in dev_str:
                            deviation_types.add('unwanted_activity')
                        elif 'existence' in dev_str or 'choice' in dev_str:
                            deviation_types.add('missing_activity')
                        else:
                            deviation_types.add('other')

                    remediation = {
                        'case_id': str(case_id),
                        'fitness': round(fitness, 4),
                        'severity': severity,
                        'deviation_count': len(deviations),
                        'deviation_types': sorted(deviation_types),
                        'recommendations': [],
                    }

                    # Generate remediation suggestions based on deviation types
                    if 'ordering' in deviation_types:
                        remediation['recommendations'].append(
                            'Check for race conditions or missing synchronization between services'
                        )
                    if 'unwanted_activity' in deviation_types:
                        remediation['recommendations'].append(
                            'Investigate unexpected retry/timeout loops in the trace path'
                        )
                    if 'missing_activity' in deviation_types:
                        remediation['recommendations'].append(
                            'Verify that required middleware or interceptor spans are not being dropped'
                        )
                    if not remediation['recommendations']:
                        remediation['recommendations'].append(
                            'Review trace for non-standard execution patterns'
                        )

                    low_fitness.append(remediation)

            # Summary report
            total_traces = len(conformance)
            total_deviating = len(low_fitness)
            critical_count = sum(1 for r in low_fitness if r['severity'] == 'critical')
            warning_count = sum(1 for r in low_fitness if r['severity'] == 'warning')

            print(f"Automated Remediation Report:")
            print(f"  Timestamp: {datetime.utcnow().isoformat()}Z")
            print(f"  Fitness threshold: {FITNESS_THRESHOLD}")
            print(f"  Total traces: {total_traces}")
            print(f"  Deviating traces: {total_deviating} ({100*total_deviating/max(total_traces,1):.1f}%)")
            print(f"    Critical (fitness < 0.5): {critical_count}")
            print(f"    Warning  (0.5 <= fitness < 0.8): {warning_count}")

            if low_fitness:
                # Aggregate recommendations
                all_recs = []
                for r in low_fitness:
                    all_recs.extend(r['recommendations'])

                from collections import Counter
                rec_counts = Counter(all_recs)

                print(f"\n  Top Remediation Actions (by frequency):")
                for rec, count in rec_counts.most_common(5):
                    print(f"    [{count:3d} traces] {rec}")

                print(f"\n  Worst traces:")
                for r in sorted(low_fitness, key=lambda x: x['fitness'])[:5]:
                    print(f"    {r['case_id'][:20]:20s} fitness={r['fitness']:.4f} severity={r['severity']:8s} devs={r['deviation_count']}")

                # Output machine-readable JSON
                report = {
                    'generated_at': datetime.utcnow().isoformat() + 'Z',
                    'fitness_threshold': FITNESS_THRESHOLD,
                    'total_traces': total_traces,
                    'deviating_traces': total_deviating,
                    'critical': critical_count,
                    'warning': warning_count,
                    'remediation_actions': rec_counts.most_common(),
                    'traces': low_fitness,
                }
                print(f"\n  JSON report (first 500 chars):")
                print(f"  {json.dumps(report, indent=2, default=str)[:500]}...")
            else:
                print("\n  All traces above fitness threshold — no remediation needed.")
    except Exception as e:
        print(f"Automated remediation failed: {e}")
else:
    print("No events for automated remediation.")

## 20. Goal-Oriented Process Mining

Compare **planned vs executed traces** to measure how faithfully agents follow their stated goals. In process mining terms:

- **The plan** — The most common trace variant represents the "intended" execution path. All other variants are deviations.
- **Plan adherence** — The fraction of traces that match the plan variant.
- **AGI alignment** — Autonomous agents have stated goals (e.g., tool sequences, task completion). Process mining validates whether execution matches intent.

This analysis is foundational for AGI safety: an agent that frequently deviates from its plan may be pursuing unintended sub-goals, encountering unexpected errors, or exhibiting emergent behavior.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        # The most common variant = "the plan" (expected behavior)
        variants = pm4py.get_variants(df)
        sorted_variants = sorted(variants.items(), key=lambda x: x[1], reverse=True)
        plan_variant = sorted_variants[0][0]
        plan_count = sorted_variants[0][1]
        total_traces = sum(v[1] for v in sorted_variants)
        
        print(f"Goal-Oriented Analysis:")
        print(f"  Plan variant (most common): {len(plan_variant)} steps")
        print(f"  Plan adherence: {plan_count}/{total_traces} traces ({100*plan_count/total_traces:.1f}%)")
        print(f"  Plan: {' → '.join(plan_variant[:8])}...")
        
        # Show deviation traces
        deviations = [(v, c) for v, c in sorted_variants[1:5]]
        if deviations:
            print(f"\n  Deviations from plan:")
            for variant, count in deviations:
                pct = 100 * count / total_traces
                print(f"    [{pct:.1f}%] {count}x: {' → '.join(variant[:6])}...")
        
        # Tool sequence compliance (if MCP tools present)
        if 'mcp.tool.name' in df.columns:
            mcp_traces = df[df['mcp.tool.name'].str.strip() != '']
            if len(mcp_traces) > 0:
                mcp_variants = pm4py.get_variants(mcp_traces)
                print(f"\n  MCP tool variant adherence: {len(mcp_variants)} variants")
    except Exception as e:
        print(f"Goal-oriented analysis failed: {e}")
else:
    print("No events for goal-oriented analysis.")

## 21. Hierarchical Process Mining

Mine **service-level process models** as hierarchy levels. Each service (`org:resource`) is a "level" with its own activity set and internal process model. Cross-service handoffs form the inter-level edges.

- **Service-level DFG** — Each service gets its own Directly-Follows Graph showing internal activity flow.
- **Cross-service handoffs** — Transitions where `org:resource` changes between consecutive events represent delegation or fan-out.
- **Hierarchy depth** — The number of distinct services involved in a trace indicates orchestration depth.

This is essential for multi-agent systems: understanding which services own which activities, and how work flows between them, reveals the implicit organizational structure of the agent swarm.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import pandas as pd

        # Per-service statistics
        services = df['org:resource'].unique()
        print(f"Hierarchical Process Mining:")
        print(f"  Distinct services (hierarchy levels): {len(services)}")

        service_stats = []
        for svc in sorted(services):
            svc_df = df[df['org:resource'] == svc]
            activities = svc_df['concept:name'].nunique()
            events = len(svc_df)
            traces = svc_df['case:concept:name'].nunique()
            avg_duration = svc_df['duration_ms'].mean() if 'duration_ms' in svc_df.columns else 0
            service_stats.append((svc, activities, events, traces, avg_duration))

        service_stats.sort(key=lambda x: x[2], reverse=True)
        print(f"\n  {'Service':40s} {'Activities':>10s} {'Events':>7s} {'Traces':>7s} {'Avg ms':>8s}")
        print(f"  {'-'*78}")
        for svc, acts, evts, trcs, dur in service_stats:
            print(f"  {svc:40s} {acts:10d} {evts:7d} {trcs:7d} {dur:8.1f}")

        # Cross-service handoffs (resource changes between consecutive events)
        df_sorted = df.sort_values(['case:concept:name', 'time:timestamp'])
        df_sorted['_next_resource'] = df_sorted.groupby('case:concept:name')['org:resource'].shift(-1)
        handoffs = df_sorted[df_sorted['org:resource'] != df_sorted['_next_resource']]
        handoff_counts = handoffs.groupby(['org:resource', '_next_resource']).size().sort_values(ascending=False)

        print(f"\n  Cross-service handoffs: {len(handoffs)}")
        print(f"  Unique handoff edges: {len(handoff_counts)}")
        if len(handoff_counts) > 0:
            print(f"\n  Top 10 handoff edges:")
            for (src, tgt), count in handoff_counts.head(10).items():
                print(f"    {str(src):35s} -> {str(tgt):35s} : {count}")

        # Hierarchy depth per trace
        depth_per_trace = df.groupby('case:concept:name')['org:resource'].nunique()
        print(f"\n  Hierarchy depth per trace:")
        print(f"    Mean: {depth_per_trace.mean():.1f} services")
        print(f"    Max:  {depth_per_trace.max()} services")
        print(f"    Min:  {depth_per_trace.min()} services")
    except Exception as e:
        print(f"Hierarchical process mining failed: {e}")
else:
    print("No events for hierarchical process mining.")

## 22. Counterfactual Analysis

Simulate **alternative execution paths** using `pm4py.play_out()` and compare them against observed behavior. This answers: *what fraction of real-world traces can the discovered model reproduce?*

- **Model coverage** — The intersection of simulated and observed variants, divided by total observed variants. High coverage means the model captures most real behavior.
  - **Emergent behavior** — Observed variants that the model *cannot* produce indicate execution paths not explained by the discovered process structure.
  - **Unrealized paths** — Simulated variants never seen in production represent potential futures or dead code paths.

For AGI systems, unreachable variants may indicate capabilities the agent has but does not exercise, or conversely, behaviors that emerged outside the model's scope.

In [ ]:
if len(df) > 0:
    try:
        import pm4py

        # Simulate 100 traces from the discovered model
        simulated_log = pm4py.play_out(net, im, fm, num_traces=100)
        sim_df = pm4py.convert_to_dataframe(simulated_log)

        # Get variant distributions
        obs_variants = pm4py.get_variants(df)
        sim_variants = pm4py.get_variants(sim_df)

        obs_set = set(obs_variants.keys())
        sim_set = set(sim_variants.keys())
        common = obs_set & sim_set
        emergent = obs_set - sim_set  # observed but model can't reproduce
        unrealized = sim_set - obs_set  # model can produce but never observed

        total_obs_traces = sum(obs_variants.values())
        covered_traces = sum(obs_variants[v] for v in common)
        coverage = covered_traces / total_obs_traces if total_obs_traces > 0 else 0

        print(f"Counterfactual Analysis:")
        print(f"  Simulated traces:  {len(sim_df)} events, {len(sim_variants)} variants")
        print(f"  Observed traces:   {len(df)} events, {len(obs_variants)} variants")
        print(f"  Model coverage:    {coverage:.1%} ({len(common)}/{len(obs_set)} variants)")
        print(f"  Emergent behavior: {len(emergent)} variants (observed but unreproducible)")
        print(f"  Unrealized paths:  {len(unrealized)} variants (simulated but never seen)")

        if emergent:
            emergent_by_freq = sorted(emergent, key=lambda v: obs_variants[v], reverse=True)
            print(f"\n  Top emergent variants (model cannot reproduce):")
            for v in emergent_by_freq[:5]:
                print(f"    [{obs_variants[v]}x] {' → '.join(v[:6])}")

        if unrealized:
            unrealized_by_freq = sorted(unrealized, key=lambda v: sim_variants[v], reverse=True)
            print(f"\n  Top unrealized paths (model can produce but never observed):")
            for v in unrealized_by_freq[:5]:
                print(f"    [{sim_variants[v]}x] {' → '.join(v[:6])}")
    except Exception as e:
        print(f"Counterfactual analysis failed: {e}")
else:
    print("No events for counterfactual analysis.")

## 23. Normative Process Mining

Treat discovered Declare constraints as **constitutional norms** — de facto rules that the process follows. Filter by confidence to distinguish enforced policies from coincidental patterns.

- **High-confidence norms** (confidence > 0) — Constraints that hold across traces, representing enforced behavioral policies.
- **Zero-confidence violations** — Activity pairs where a constraint template applies but is never satisfied, indicating potential norm violations or policy gaps.
- **Support threshold** — Only constraints supported by at least `min_support_ratio` fraction of traces are considered, filtering noise.

For AGI governance, normative mining reveals the actual behavioral constitution of the system — what rules the agent *de facto* follows, independent of any stated policy.

In [ ]:
if len(df) > 0:
    try:
        import pm4py

        # Discover Declare constraints as norms
        declare_model = pm4py.discover_declare(df, min_support_ratio=0.5)

        # Classify norms by confidence
        high_confidence = []  # enforced policies
        zero_confidence = []  # potential violations
        template_summary = {}

        for tpl, instances in declare_model.items():
            tpl_name = tpl if isinstance(tpl, str) else getattr(tpl, 'name', str(tpl))
            template_summary[tpl_name] = template_summary.get(tpl_name, 0) + 1

            for key, metrics in instances.items():
                # key can be a string (single activity) or tuple (activity pair)
                key_str = str(key)
                confidence = metrics.get('confidence', 0) if isinstance(metrics, dict) else 0
                support = metrics.get('support', 0) if isinstance(metrics, dict) else 0

                entry = {
                    'template': tpl_name,
                    'key': key_str,
                    'confidence': confidence,
                    'support': support,
                }

                if confidence > 0:
                    high_confidence.append(entry)
                else:
                    zero_confidence.append(entry)

        high_confidence.sort(key=lambda x: x['confidence'], reverse=True)

        print(f"Normative Process Mining:")
        print(f"  Total constraints: {len(declare_model)}")
        print(f"  High-confidence norms (enforced): {len(high_confidence)}")
        print(f"  Zero-confidence pairs (violations): {len(zero_confidence)}")
        print(f"\n  Template distribution:")
        for tpl, count in sorted(template_summary.items(), key=lambda x: x[1], reverse=True):
            print(f"    {tpl}: {count}")

        if high_confidence:
            print(f"\n  Top 10 enforced norms:")
            for entry in high_confidence[:10]:
                print(f"    [{entry['confidence']:.2f}] {entry['template']}: {entry['key']}")

        if zero_confidence:
            print(f"\n  Potential norm violations (zero confidence, top 10):")
            for entry in zero_confidence[:10]:
                print(f"    [0.00] {entry['template']}: {entry['key']}")
    except Exception as e:
        print(f"Normative process mining failed: {e}")
else:
    print("No events for normative process mining.")

## 24. Multi-Agent Orchestration

Mine the **delegation protocol** between services. For each trace, the sequence of services (`org:resource`) forms a "delegation pattern" — who calls whom, and in what order.

- **Delegation patterns** — The sequence of services involved in each trace, analogous to a workflow routing plan.
- **Service-level DFG** — A Directly-Follows Graph at the service level (not activity level), showing handoff topology.
- **Orchestration patterns** — Common delegation sequences reveal the implicit coordination protocol between agents.

This is the process-mining view of multi-agent orchestration: instead of looking at *what* activities happen, we look at *who* performs them and *how work is delegated* across the agent swarm.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import pandas as pd
        from collections import Counter

        # Extract service sequence per trace (delegation pattern)
        df_sorted = df.sort_values(['case:concept:name', 'time:timestamp'])
        delegation_patterns = []

        for case_id, group in df_sorted.groupby('case:concept:name'):
            # Get unique service sequence (preserve order, remove consecutive duplicates)
            services = list(group['org:resource'])
            pattern = []
            prev = None
            for svc in services:
                if svc != prev:
                    pattern.append(svc)
                    prev = svc
            delegation_patterns.append(tuple(pattern))

        pattern_counts = Counter(delegation_patterns)
        sorted_patterns = pattern_counts.most_common(10)

        print(f"Multi-Agent Orchestration:")
        print(f"  Total traces: {len(delegation_patterns)}")
        print(f"  Unique delegation patterns: {len(pattern_counts)}")

        if sorted_patterns:
            print(f"\n  Top 10 orchestration patterns:")
            for i, (pattern, count) in enumerate(sorted_patterns):
                pct = 100 * count / len(delegation_patterns)
                seq = ' → '.join(pattern[:8])
                if len(pattern) > 8:
                    seq += '...'
                print(f"    {i+1}. [{pct:5.1f}%] {count:4d}x: {seq}")

        # Service-level DFG (handoff topology)
        svc_dfg = Counter()
        for pattern in delegation_patterns:
            for j in range(len(pattern) - 1):
                svc_dfg[(pattern[j], pattern[j+1])] += 1

        print(f"\n  Service-level DFG ({len(svc_dfg)} edges):")
        for (src, tgt), count in svc_dfg.most_common(10):
            print(f"    {str(src):35s} -> {str(tgt):35s} : {count}")

        # Orchestration depth distribution
        depths = [len(p) for p in delegation_patterns]
        print(f"\n  Orchestration depth (services per trace):")
        print(f"    Mean: {pd.Series(depths).mean():.1f}")
        print(f"    Max:  {max(depths)}")
        print(f"    Min:  {min(depths)}")
    except Exception as e:
        print(f"Multi-agent orchestration analysis failed: {e}")
else:
    print("No events for multi-agent orchestration analysis.")

## 25. Self-Reflective Process Mining

Combine Declare conformance checking with **policy generation** to close the meta-loop: process mining output feeds back into agent behavior.

- **Fitness classification** — Traces are classified by their Declare fitness score into tiers (high/medium/low).
- **Most violated templates** — The constraint templates that are violated most frequently reveal systemic weaknesses.
- **Policy recommendations** — Each violation pattern maps to an actionable policy recommendation that can be injected into the agent's control loop.

This is the **self-reflective** layer: the agent examines its own execution traces, identifies behavioral anti-patterns, and generates governance policies to prevent recurrence. It is the process-mining analogue of an agent reading its own audit log and deciding to change its behavior.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        from collections import Counter

        # Step 1: Discover constraints and check conformance
        declare_model = pm4py.discover_declare(df, min_support_ratio=0.5)
        conformance = pm4py.conformance_declare(df, declare_model)

        if not conformance:
            print("Self-Reflective Analysis: all traces satisfy all constraints.")
            print("No policy recommendations needed.")
        else:
            # Step 2: Classify traces by fitness
            high_fitness = []
            med_fitness = []
            low_fitness = []
            all_deviations = []

            for i, result in enumerate(conformance):
                if not isinstance(result, dict):
                    continue
                fit = result.get('dev_fitness', 1.0)
                devs = result.get('deviations', [])

                if fit >= 0.9:
                    high_fitness.append(result)
                elif fit >= 0.5:
                    med_fitness.append(result)
                else:
                    low_fitness.append(result)

                for d in devs:
                    all_deviations.append(str(d))

            total = len(conformance)
            print(f"Self-Reflective Process Mining:")
            print(f"  Total traces: {total}")
            print(f"  High fitness (>=0.9):   {len(high_fitness):4d} ({100*len(high_fitness)/total:.1f}%)")
            print(f"  Medium fitness (0.5-0.9): {len(med_fitness):4d} ({100*len(med_fitness)/total:.1f}%)")
            print(f"  Low fitness (<0.5):     {len(low_fitness):4d} ({100*len(low_fitness)/total:.1f}%)")

            # Step 3: Extract most violated templates
            violation_templates = Counter()
            for dev in all_deviations:
                dev_lower = dev.lower()
                for tpl in ['response', 'precedence', 'succession', 'absence', 'existence', 'choice', 'init', 'negation']:
                    if tpl in dev_lower:
                        violation_templates[tpl] += 1
                        break

            if violation_templates:
                print(f"\n  Most violated templates:")
                for tpl, count in violation_templates.most_common(10):
                    print(f"    {tpl}: {count} violations")

            # Step 4: Generate policy recommendations
            recommendations = []
            if violation_templates.get('response', 0) > 0:
                recommendations.append(
                    "POLICY: Enforce response pairs — every request activity must be followed by its corresponding response within a timeout."
                )
            if violation_templates.get('precedence', 0) > 0:
                recommendations.append(
                    "POLICY: Enforce precedence ordering — prerequisite activities must complete before dependent activities begin."
                )
            if violation_templates.get('absence', 0) > 0:
                recommendations.append(
                    "POLICY: Enforce absence constraints — forbidden activities must not appear in any trace."
                )
            if violation_templates.get('existence', 0) > 0:
                recommendations.append(
                    "POLICY: Enforce existence constraints — required activities must appear at least once per trace."
                )
            if len(low_fitness) > total * 0.1:
                recommendations.append(
                    f"ALERT: {100*len(low_fitness)/total:.0f}% of traces have low fitness (<0.5). Investigate systemic issues."
                )
            if not recommendations:
                recommendations.append(
                    "No policy actions needed — process execution is within acceptable bounds."
                )

            print(f"\n  Policy Recommendations ({len(recommendations)}):")
            for i, rec in enumerate(recommendations, 1):
                print(f"    {i}. {rec}")
    except Exception as e:
        print(f"Self-reflective process mining failed: {e}")
else:
    print("No events for self-reflective process mining.")

## 8. Temporal Profile Conformance
Check SLA compliance at the process level — compare observed inter-activity times against expected temporal profiles.

In [ ]:
if len(df) > 0:
    try:
        import numpy as np
        from sklearn.ensemble import IsolationForest
        from sklearn.preprocessing import StandardScaler
        from sklearn.model_selection import train_test_split

        # Feature extraction per trace
        case_features = df.groupby('case:concept:name').agg(
            span_count=('concept:name', 'count'),
            unique_services=('org:resource', 'nunique'),
            total_duration_ms=('duration_ms', 'sum'),
            mean_duration_ms=('duration_ms', 'mean'),
            max_duration_ms=('duration_ms', 'max'),
            std_duration_ms=('duration_ms', 'std'),
            has_mcp_tool=('mcp.tool.name', lambda x: int((x.str.strip() != '').sum())),
        ).fillna(0)

        # Label anomalies: traces with duration > 2 std from mean
        mean_dur = case_features['total_duration_ms'].mean()
        std_dur = case_features['total_duration_ms'].std()
        case_features['is_anomaly_label'] = (
            (case_features['total_duration_ms'] > mean_dur + 2 * std_dur) |
            (case_features['total_duration_ms'] < mean_dur - 2 * std_dur)
        ).astype(int)

        # Scale features
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(case_features)

        # Isolation Forest baseline
        clf = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
        case_features['anomaly_score'] = clf.decision_function(features_scaled)
        case_features['is_anomaly'] = clf.fit_predict(features_scaled)
        anomalies_if = case_features[case_features['is_anomaly'] == -1]

        print("Baseline (Isolation Forest):")
        print(f"  Anomalous: {len(anomalies_if)}/{len(case_features)} ({100*len(anomalies_if)/len(case_features):.1f}%)")

        # TPOT2 AutoML
        n_anomalous = case_features['is_anomaly_label'].sum()
        n_normal = len(case_features) - n_anomalous

        if n_anomalous >= 3 and n_normal >= 5 and len(case_features) >= 20:
            try:
                from tpot2 import TPOTClassifier

                print(f"\nRunning TPOT2 AutoML (labeled anomalies: {n_anomalous}, normal: {n_normal})...")

                tpot = TPOTClassifier(
                    generations=5,
                    population_size=20,
                    cv=3,
                    scoring='f1',
                    verbosity=0,
                    random_state=42,
                    max_time_mins=1,
                    n_jobs=-1,
                )
                tpot.fit(features_scaled, case_features['is_anomaly_label'].values)

                best_score = tpot.score(features_scaled, case_features['is_anomaly_label'].values)
                best_pipeline = str(tpot.fitted_pipeline_)

                print(f"  Best F1 Score: {best_score:.4f}")
                print(f"  Best Pipeline: {best_pipeline[:200]}")

                # Export the pipeline
                tpot.export('tpot2_best_pipeline.py')
                print(f"  Pipeline exported to: tpot2_best_pipeline.py")
            except ImportError:
                print("TPOT2 not installed: pip install tpot2")
            except Exception as e:
                print(f"TPOT2 failed: {e}")
        else:
            print(f"\nTPOT2 skipped: insufficient labeled anomalies ({n_anomalous}) or samples ({len(case_features)})")
    except ImportError:
        print("Anomaly detection requires scikit-learn: pip install scikit-learn")
    except Exception as e:
        print(f"Anomaly detection failed: {e}")
else:
    print("No events for TPOT2 analysis.")

## 15. TPOT2 AutoML Anomaly Detection
Use TPOT2 to automatically find the best ML pipeline for trace anomaly classification.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        from datetime import timedelta
        
        df_sorted = df.sort_values('time:timestamp')
        min_ts = df_sorted['time:timestamp'].min()
        max_ts = df_sorted['time:timestamp'].max()
        total_hours = (max_ts - min_ts).total_seconds() / 3600
        
        if total_hours < 1:
            print(f"Insufficient time range ({total_hours:.1f}h) for drift detection")
        else:
            # Split into two equal time windows
            midpoint = min_ts + timedelta(hours=total_hours / 2)
            df_early = df_sorted[df_sorted['time:timestamp'] < midpoint]
            df_late = df_sorted[df_sorted['time:timestamp'] >= midpoint]
            
            # Discover variants per window
            variants_early = pm4py.get_variants(df_early)
            variants_late = pm4py.get_variants(df_late)
            
            common = set(variants_early.keys()) & set(variants_late.keys())
            only_early = set(variants_early.keys()) - set(variants_late.keys())
            only_late = set(variants_late.keys()) - set(variants_early.keys())
            
            print("Process Drift Detection:")
            print(f"  Window 1 (first {total_hours/2:.1f}h): {len(df_early)} events, {len(variants_early)} variants")
            print(f"  Window 2 (last {total_hours/2:.1f}h):  {len(df_late)} events, {len(variants_late)} variants")
            print(f"  Common variants: {len(common)}")
            print(f"  New variants: {len(only_late)}")
            print(f"  Disappeared variants: {len(only_early)}")
            
            if only_late:
                print(f"\n  New patterns (top 5):")
                for v in sorted(only_late, key=lambda x: variants_late[x], reverse=True)[:5]:
                    print(f"    - {' → '.join(v)}")
            
            if only_early:
                print(f"\n  Disappeared patterns (top 5):")
                for v in sorted(only_early, key=lambda x: variants_early[x], reverse=True)[:5]:
                    print(f"    - {' → '.join(v)}")
            
            if not only_late and not only_early:
                print("\n  No drift detected — variant distribution is stable")
    except Exception as e:
        print(f"Drift detection failed: {e}")
else:
    print("No events for drift detection.")

## 14. Process Drift Detection
Compare process trees across time windows to detect operational drift.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        
        # Discover stochastic Petri net
        snet, sim, sfm = pm4py.discover_stochastic_petri_net(df)
        
        print("Stochastic Process Model:")
        print(f"  Places: {len(snet.places)}")
        print(f"  Transitions: {len(snet.transitions)}")
        print(f"  Arcs: {len(snet.arcs)}")
        
        # Show transitions with probabilities
        prob_transitions = []
        for t in snet.transitions:
            prob = None
            if hasattr(t, 'properties') and 'probability' in t.properties:
                prob = float(t.properties['probability'])
            if prob is not None:
                prob_transitions.append((t.name, prob))
        
        prob_transitions.sort(key=lambda x: x[1], reverse=True)
        print(f"\n  Top 10 transitions by probability:")
        for name, prob in prob_transitions[:10]:
            print(f"    {name:50s} P={prob:.4f}")
    except Exception as e:
        print(f"Stochastic mining failed: {e}")
else:
    print("No events for stochastic mining.")

## 13. Stochastic Process Mining
Discover stochastic Petri net with transition probabilities.

In [ ]:
if len(df) > 0:
    try:
        import numpy as np
        from sklearn.ensemble import IsolationForest
        from sklearn.preprocessing import StandardScaler
        
        # Feature extraction per trace
        case_features = df.groupby('case:concept:name').agg(
            span_count=('concept:name', 'count'),
            unique_services=('org:resource', 'nunique'),
            total_duration_ms=('duration_ms', 'sum'),
            mean_duration_ms=('duration_ms', 'mean'),
            max_duration_ms=('duration_ms', 'max'),
            std_duration_ms=('duration_ms', 'std'),
            has_mcp_tool=('mcp.tool.name', lambda x: int(x.str.strip() != '').sum()),
        ).fillna(0)
        
        # Scale features
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(case_features)
        
        # Fit Isolation Forest (10% contamination = expect 10% anomalies)
        clf = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
        case_features['anomaly_score'] = clf.decision_function(features_scaled)
        case_features['is_anomaly'] = clf.predict(features_scaled)
        
        anomalies = case_features[case_features['is_anomaly'] == -1]
        
        print("ML-Based Anomaly Detection:")
        print(f"  Total cases:       {len(case_features)}")
        print(f"  Anomalous cases:   {len(anomalies)} ({100*len(anomalies)/len(case_features):.1f}%)")
        print(f"  Anomaly score range: [{case_features['anomaly_score'].min():.3f}, {case_features['anomaly_score'].max():.3f}]")
        
        if len(anomalies) > 0:
            print(f"\n  Top 5 anomalous traces (by span count):")
            for i, (trace_id, row) in enumerate(anomalies.nlargest(5, 'span_count').iterrows()):
                print(f"    {i+1}. {trace_id[:16]}... score={row['anomaly_score']:.3f} spans={int(row['span_count'])} duration={row['total_duration_ms']:.0f}ms")
    except ImportError:
        print("Anomaly detection requires scikit-learn: pip install scikit-learn")
    except Exception as e:
        print(f"Anomaly detection failed: {e}")
else:
    print("No events for anomaly detection.")

## 12. ML-Based Anomaly Detection
Use feature extraction + Isolation Forest to detect anomalous traces.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        import numpy as np
        
        # Compute alignments — identifies exact deviations per trace
        aligned_traces = pm4py.conformance_diagnostics_alignments(df, net, im, fm)
        
        if aligned_traces:
            fitnesses = [a['fitness'] for a in aligned_traces if 'fitness' in a]
            costs = [a.get('cost', 0) for a in aligned_traces]
            
            print("Alignment-Based Conformance:")
            print(f"  Mean Fitness: {np.mean(fitnesses):.4f}")
            print(f"  Min Fitness:  {np.min(fitnesses):.4f}")
            print(f"  Mean Cost:    {np.mean(costs):.2f}")
            print(f"  Deviating (fitness < 1.0): {sum(1 for f in fitnesses if f < 1.0)}/{len(fitnesses)}")
            
            # Show top 5 most deviating traces
            deviating = sorted(
                [a for a in aligned_traces if a.get('fitness', 1.0) < 1.0],
                key=lambda x: x.get('cost', 0),
                reverse=True
            )[:5]
            
            if deviating:
                print("\n  Top 5 deviations:")
                for i, d in enumerate(deviating):
                    print(f"    {i+1}. fitness={d['fitness']:.4f}, cost={d.get('cost', 'N/A')}")
        else:
            print("No alignment results")
    except Exception as e:
        print(f"Alignment conformance failed: {e}")
else:
    print("No events for alignment conformance.")

## 11. Alignment-Based Conformance Checking
Precise per-trace deviation diagnosis with alignment costs.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        
        # Convert traces to OCEL — services and MCP tools as object types
        ocel = pm4py.convert_log_to_ocel(
            df,
            object_types=['org:resource', 'mcp.tool.name'],
            activity_column='concept:name',
            timestamp_column='time:timestamp',
            case_id_column='case:concept:name',
        )
        
        # Discover object-centric Petri net
        ocpn = pm4py.discover_oc_petri_net(ocel)
        
        # Flatten OCEL for inspection
        ocel_df = pm4py.ocel_flattening(ocel)
        
        print(f"OCEL Events: {len(ocel_df)}")
        print(f"Object Types: {list(ocel.object_types) if hasattr(ocel, 'object_types') else 'N/A'}")
        print(f"OCPN Places: {len(ocpn.places)}")
        print(f"OCPN Transitions: {len(ocpn.transitions)}")
        print(f"OCPN Arcs: {len(ocpn.arcs)}")
        
        # View object-centric Petri net
        pm4py.view_ocpn(ocpn)
    except Exception as e:
        print(f"Object-centric mining failed: {e}")
else:
    print("No events for OCEL analysis.")

## 10. Object-Centric Process Mining (OCEL)
Discover object-centric Petri net preserving multi-service relationships.

In [ ]:
if len(df) > 0:
    try:
        import pm4py
        
        # Predict remaining time for each trace
        pred_time = pm4py.predict_remaining_time(df)
        if pred_time is not None and len(pred_time) > 0:
            print("Remaining Time Prediction (ms):")
            print(f"  Mean:   {pred_time.mean():.1f}")
            print(f"  Median: {pred_time.median():.1f}")
            print(f"  Max:    {pred_time.max():.1f}")
            print(f"  Std:    {pred_time.std():.1f}")
        else:
            print("No remaining time predictions generated (insufficient data)")
    except Exception as e:
        print(f"Remaining time prediction failed: {e}")
    
    try:
        import pm4py
        
        # Predict next activity
        pred_next = pm4py.predict_next_activity(df)
        if pred_next is not None:
            print(f"\nNext Activity Prediction available: {len(pred_next)} prefixes")
        else:
            print("\nNo next activity predictions generated")
    except Exception as e:
        print(f"Next activity prediction failed: {e}")
else:
    print("No events for predictive monitoring.")

## 9. Predictive Monitoring — Remaining Time & Next Activity
Predict remaining trace latency and next span for in-flight traces.

In [ ]:
if len(df) > 0:
    import pm4py
    
    # Get all trace variants (unique activity sequences)
    variants = pm4py.get_variants(df)
    
    # Sort by frequency
    sorted_variants = sorted(variants.items(), key=lambda x: x[1], reverse=True)
    
    total_traces = sum(v[1] for v in sorted_variants)
    
    print(f"Total variants: {len(sorted_variants)}")
    print(f"Total traces: {total_traces}")
    print(f"\nTop 10 variants:")
    for i, (variant, count) in enumerate(sorted_variants[:10]):
        pct = 100 * count / total_traces
        print(f"  {i+1}. [{pct:5.1f}%] {count:4d}x: {' → '.join(variant)}")
else:
    print("No events for variant analysis.")

## 8. Variant Analysis
Identify the most common span execution patterns (trace variants).